In [1]:
from aicspylibczi import CziFile
import numpy as np
from bioio_ome_zarr.writers import Channel, OMEZarrWriter
import dask.array as da
from zarr.codecs import BloscCodec
from typing import Dict, List, Optional
import dask
import skimage as sk
from bioio import BioImage

In [4]:
def calculate_pyramid_shapes(
    base_shape: tuple,
    num_channels: int,
    num_levels: int = 5,
    downsample_factors: Optional[List[int]] = None
) -> List[tuple]:
    """
    Calculate shapes for each pyramid level.

    Parameters
    ----------
    base_shape : tuple
        Shape of the full resolution image (Y, X)
    num_channels : int
        Number of channels
    num_levels : int, optional
        Number of pyramid levels (default: 5)
    downsample_factors : list of int, optional
        Custom downsample factors (default: [1, 2, 4, 8, 12])

    Returns
    -------
    list of tuple
        List of shapes (C, Y, X) for each pyramid level
    """
    if downsample_factors is None:
        downsample_factors = [1, 2, 4, 8, 12]

    # Ensure we have the right number of factors
    if len(downsample_factors) != num_levels:
        # Generate factors
        downsample_factors = [2**i if i < 4 else 12 for i in range(num_levels)]

    level_shapes = []
    for factor in downsample_factors:
        y_size = int(base_shape[0] / factor)
        x_size = int(base_shape[1] / factor)
        level_shapes.append((num_channels, y_size, x_size))

    return level_shapes

In [7]:
tma = CziFile(r'TMArescan_03122026\TMA_triplestain_rescan.czi')
metadata = tma.read_subblock_metadata(unified_xml=True)

In [ ]:
tma_img = dtma.read_image()

In [ ]:
pixel_sizes = first_img.physical_pixel_sizes
physical_pixel_size = [
    0,  # C dimension has no physical size
    pixel_sizes.Y if pixel_sizes.Y else 1.0,
    pixel_sizes.X if pixel_sizes.X else 1.0
]

print(f"  Physical pixel size: Y={physical_pixel_size[1]}, X={physical_pixel_size[2]} µm")

# Calculate pyramid level shapes
base_shape = image_stack.shape[1:]  # (Y, X)
level_shapes = calculate_pyramid_shapes(
    base_shape,
    len(tiff_files),
    num_levels=pyramid_levels
)

In [ ]:
writer = OMEZarrWriter(
        store=str(output_path),
        level_shapes=level_shapes,
        dtype=image_stack.dtype,
        zarr_format=2,
        channels=channel_objects,
        axes_names=["c", "y", "x"],
        axes_types=["channel", "space", "space"],
        axes_units=[None, "micrometer", "micrometer"],
        physical_pixel_size=physical_pixel_size
    )

    # Write the data
print(f"  Writing data to Zarr (this may take a while)...")
writer.write_full_volume(image_stack)